# 🎯 Day 09 — Mock Interview Practice

**Week 2 · Day 2 — INTERMEDIATE → ADVANCED** | 18 questions · 8 categories | Oct 21, 2026

Continuing the Week-2 ramp-up (Day 08 covered sessionization, cohort retention, LRU cache, BFS, CDC MERGE, AQE internals, exactly-once, SCD 3/4/6, zero-copy cloning, 3 AM debugging). Today: recursive CTEs, time-series gap filling, decorators, Kahn's algorithm, streak detection, idempotent incremental loads, join-strategy tuning, GC & memory management, DQ gates, backpressure, circuit breakers, fact-table types, Data Vault 2.0, Snowflake Streams+Tasks, Row Access Policies, structured logging, cost attribution, and streaming-app monitoring.

| # | Category | Topic | Level |
| --- | --- | --- | --- |
| Q1 | SQL Advanced | Recursive CTE — org-chart hierarchy | INTERMEDIATE |
| Q2 | SQL Advanced | Time-series gaps — calendar spine + fill | ADVANCED |
| Q3 | Python | Decorators — @timed, @retry, functools.wraps | INTERMEDIATE |
| Q4 | Python | Topological sort — Kahn's algorithm | ADVANCED |
| Q5 | PySpark Coding | Consecutive-day activity streaks | INTERMEDIATE |
| Q6 | PySpark Coding | Idempotent incremental loads — MERGE vs replaceWhere | ADVANCED |
| Q7 | Spark Optimization | Join strategy selection — reading the physical plan | INTERMEDIATE |
| Q8 | Spark Optimization | GC tuning & executor memory management | ADVANCED |
| Q9 | ETL | Data-quality gate + quarantine (expectations pattern) | INTERMEDIATE |
| Q10 | ETL | Backpressure & rate limiting in streaming | ADVANCED |
| Q11 | ETL | Circuit breaker for upstream dependencies | ADVANCED |
| Q12 | Data Modeling | Fact-table types — transaction / periodic / accumulating | INTERMEDIATE |
| Q13 | Data Modeling | Data Vault 2.0 — hubs, links, satellites | ADVANCED |
| Q14 | Snowflake | Streams + Tasks — incremental CDC pipelines | INTERMEDIATE |
| Q15 | Snowflake | Row Access Policies vs secure views | ADVANCED |
| Q16 | Monitoring & Logging | Structured logging with run context | INTERMEDIATE |
| Q17 | Monitoring & Logging | Cost attribution & chargeback via system tables | ADVANCED |
| Q18 | Monitoring & Logging | Monitoring Structured Streaming apps | ADVANCED |

*How to practice: read the problem, attempt it yourself in a scratch cell, then study the solution. 7 INTERMEDIATE / 11 ADVANCED today.*

## Q1 [INTERMEDIATE] SQL — Recursive CTE: Org-Chart Hierarchy Traversal

**Company scenario (Uber-style):** You have an `employees` table with `emp_id`, `name`, `manager_id` (`NULL` for the CEO). HR asks two things:

1. For every employee: their **depth** in the org chart and the full **management chain** from the CEO down to them, e.g. `Priya (CEO) > Alice > Carol > Eve`.
2. For every manager: how many people report to them **directly OR indirectly**.

**Requirements:**

* Use a recursive CTE (`WITH RECURSIVE`, supported in Databricks Runtime 17+).
* Output columns for part 1: `emp_id, name, manager_id, depth, chain`.
* Output columns for part 2: `emp_id, direct_reports, total_reports`.

**Sample data** (6 employees, 4 levels — Eve is 3 levels below the CEO):

| emp_id | name | manager_id |
| --- | --- | --- |
| 1 | Priya (CEO) | NULL |
| 2 | Alice | 1 |
| 3 | Bob | 1 |
| 4 | Carol | 2 |
| 5 | Dan | 2 |
| 6 | Eve | 4 |

**Follow-ups an interviewer may ask:** What if `manager_id` forms a cycle? What are Spark's recursion limits (100 iterations / 1M rows by default)? How did engines handle this before recursive CTE support?

In [0]:
# Solution Q1 — Recursive CTE for hierarchy traversal
from pyspark.sql import functions as F, types as T

schema = T.StructType([
    T.StructField("emp_id", T.IntegerType()),
    T.StructField("name", T.StringType()),
    T.StructField("manager_id", T.IntegerType()),
])
employees = spark.createDataFrame([
    (1, "Priya (CEO)", None),
    (2, "Alice", 1),
    (3, "Bob", 1),
    (4, "Carol", 2),
    (5, "Dan", 2),
    (6, "Eve", 4),
], schema)
employees.createOrReplaceTempView("employees")

# --- Part 1: depth + management chain (anchor = CEO, recursive = join on manager) ---
part1 = spark.sql("""
WITH RECURSIVE org_tree AS (
  -- Anchor: the root of the tree (CEO has no manager)
  SELECT emp_id, name, manager_id,
         0 AS depth,
         CAST(name AS STRING) AS chain
  FROM employees
  WHERE manager_id IS NULL

  UNION ALL

  -- Recursive member: every employee whose manager is already in the tree
  SELECT e.emp_id, e.name, e.manager_id,
         t.depth + 1,
         CONCAT(t.chain, ' > ', e.name)
  FROM employees e
  JOIN org_tree t ON e.manager_id = t.emp_id
)
SELECT emp_id, name, manager_id, depth, chain
FROM org_tree
ORDER BY emp_id
""")
part1.show(truncate=False)

# --- Part 2: direct + indirect reports (second recursive CTE over descendants) ---
part2 = spark.sql("""
WITH RECURSIVE descendants AS (
  -- Every employee is a member of their own subtree
  SELECT emp_id AS root_id, emp_id AS member_id FROM employees

  UNION ALL

  -- Recursively add each member's direct reports to the root's subtree
  SELECT d.root_id, e.emp_id
  FROM descendants d
  JOIN employees e ON e.manager_id = d.member_id
),
direct AS (                                -- pre-aggregated: one row per manager
  SELECT manager_id AS mgr, COUNT(*) AS direct_reports
  FROM employees
  WHERE manager_id IS NOT NULL
  GROUP BY manager_id
),
total AS (                                 -- pre-aggregated: one row per subtree root
  SELECT root_id, COUNT(*) - 1 AS total_reports
  FROM descendants
  GROUP BY root_id
)
SELECT
  e.emp_id,
  e.name,
  COALESCE(d.direct_reports, 0) AS direct_reports,
  COALESCE(t.total_reports, 0) AS total_reports
FROM employees e
LEFT JOIN direct d ON d.mgr = e.emp_id
LEFT JOIN total  t ON t.root_id = e.emp_id
ORDER BY e.emp_id
""")
part2.show()

# --- Verify ---
rows = {r["emp_id"]: r for r in part1.collect()}
assert rows[6]["depth"] == 3
assert rows[6]["chain"] == "Priya (CEO) > Alice > Carol > Eve"
assert rows[2]["depth"] == 1
rep = {r["emp_id"]: r for r in part2.collect()}
assert rep[2]["total_reports"] == 3          # Carol, Dan, Eve
assert rep[2]["direct_reports"] == 2
assert rep[4]["direct_reports"] == 1 and rep[4]["total_reports"] == 1
assert rep[1]["total_reports"] == 5          # everyone except the CEO
print("✅ Q1 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. WITH RECURSIVE = anchor (base case) UNION ALL recursive member that references the CTE itself.
# 2. Spark guards: default 100 iterations / 1M rows; override with MAX RECURSION LEVEL / LIMIT ALL.
# 3. Cycle prevention: array_append a visited-path column and filter NOT array_contains(path, node).
# 4. Pre-DBR-17 fallback: iterative self-join loop in PySpark (while frontier not empty) — same idea, manual bookkeeping.
# 5. Counting across TWO 1:N joins fans out rows (2 direct x 4 subtree = 8 before you know it):
#    pre-aggregate each side in its own CTE and LEFT JOIN the 1-row summaries — never
#    COUNT over a fanned-out join.

## Q2 [ADVANCED] SQL — Time-Series Gaps: Calendar Spine + Gap Filling

**Company scenario (Netflix-style):** `daily_revenue(store, d, revenue)` only has rows for days a store **actually reported** — zero-sale days are simply missing. Finance needs a **complete daily series per store** with:

1. Missing days filled with `revenue = 0` (marked `reported = false`).
2. A **rolling 7-day revenue** computed on the filled, contiguous series.

**Sample data** (2026-10-01 → 2026-10-07, 2 stores, holes everywhere):

| store | d | revenue |
| --- | --- | --- |
| S1 | 10-01 | 100 |
| S1 | 10-03 | 120 |
| S1 | 10-07 | 90 |
| S2 | 10-01 | 300 |
| S2 | 10-02 | 310 |
| S2 | 10-05 | 280 |

**Expected:** 14 output rows (7 days × 2 stores), e.g. S1 on 10-02 = 0 with `reported = false`, and the rolling 7-day sum for S1 on 10-03 = 100 + 0 + 120 = 220.

**Follow-ups:** Why can't you gap-fill with `LAG`/`LEAD` alone? How do you handle a store that never reported in the range? What about hourly granularity?

In [0]:
# Solution Q2 — Calendar spine + gap fill + rolling window
from pyspark.sql import functions as F, types as T

rev_schema = T.StructType([
    T.StructField("store", T.StringType()),
    T.StructField("d", T.StringType()),
    T.StructField("revenue", T.DoubleType()),
])
rev = (spark.createDataFrame([
    ("S1", "2026-10-01", 100.0), ("S1", "2026-10-03", 120.0), ("S1", "2026-10-07", 90.0),
    ("S2", "2026-10-01", 300.0), ("S2", "2026-10-02", 310.0), ("S2", "2026-10-05", 280.0),
], rev_schema)
       .withColumn("d", F.to_date("d")))
rev.createOrReplaceTempView("daily_revenue")

result = spark.sql("""
WITH bounds AS (                        -- derive the date range from the data itself
  SELECT MIN(d) AS lo, MAX(d) AS hi FROM daily_revenue
),
calendar AS (                            -- one row per day in the range
  SELECT explode(sequence(lo, hi, INTERVAL 1 DAY)) AS d FROM bounds
),
spine AS (                               -- every day × every store = complete grid
  SELECT c.d, s.store
  FROM calendar c
  CROSS JOIN (SELECT DISTINCT store FROM daily_revenue) s
),
filled AS (                              -- left join, coalesce missing revenue to 0
  SELECT sp.d, sp.store,
         COALESCE(r.revenue, 0) AS revenue,
         (r.d IS NOT NULL) AS reported
  FROM spine sp
  LEFT JOIN daily_revenue r
    ON sp.d = r.d AND sp.store = r.store
)
SELECT
  store, d, revenue, reported,
  SUM(revenue) OVER (
    PARTITION BY store ORDER BY d
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS rolling_7d
FROM filled
ORDER BY store, d
""")
result.show()

# --- Verify ---
assert result.count() == 14
rows = {(r["store"], str(r["d"])): r for r in result.collect()}
s1_02 = rows[("S1", "2026-10-02")]
assert float(s1_02["revenue"]) == 0.0 and s1_02["reported"] is False
s1_03 = rows[("S1", "2026-10-03")]
assert float(s1_03["rolling_7d"]) == 220.0          # 100 + 0 + 120
print("✅ Q2 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Pattern = calendar spine (sequence + explode) CROSS JOIN dimension, then LEFT JOIN + COALESCE.
# 2. LAG/LEAD compare neighbors — they do NOT generate the missing rows; you need the spine first.
# 3. For irregular intervals (validity ranges), use a range join: spine JOIN events ON d BETWEEN start AND end.
# 4. At scale, materialize the calendar once (date-dim table) instead of exploding per query.

## Q3 [INTERMEDIATE] Python — Decorators: `@timed`, `@retry`, and `functools.wraps`

**Company scenario:** In your ETL codebase every external call needs (a) **timing** for the ops dashboard and (b) **retry with exponential backoff** for transient 5xx failures. Build both as reusable decorators, then stack them.

**Requirements:**

* `timed(fn)` — prints `<name> took X ms` for every call, via `time.perf_counter()`.
* `retry(times=3, exceptions=(...), backoff=0.0)` — configurable decorator *factory*; sleeps `backoff * 2^(attempt-1)` between tries; re-raises the last exception when exhausted.
* Both must use `@functools.wraps` so the wrapped function's `__name__`/`__doc__` survive (critical for logging and introspection).
* Test on a `flaky_fetch` function that raises `RuntimeError` twice, then succeeds — verify result, call count, and that `__name__` is preserved.

**Follow-ups:** Why is decorator order (`@timed` outside `@retry` vs inside) different? What breaks if you skip `functools.wraps`?

In [0]:
# Solution Q3 — Decorators: @timed + @retry with functools.wraps
import functools, time

def timed(fn):
    """Log wall-clock duration of every call (finally => also times failures)."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            ms = (time.perf_counter() - t0) * 1000
            print(f"[timed] {fn.__name__} took {ms:.3f} ms")
    return wrapper

def retry(times=3, exceptions=(Exception,), backoff=0.0):
    """Decorator factory: retry on selected exception types with exponential backoff."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    last = exc
                    print(f"[retry] {fn.__name__} attempt {attempt}/{times} failed: {exc}")
                    if attempt < times and backoff > 0:
                        time.sleep(backoff * (2 ** (attempt - 1)))
            raise last
        return wrapper
    return decorator

# --- Flaky upstream: fails twice, then succeeds ---
_state = {"calls": 0}

@timed
@retry(times=3, exceptions=(RuntimeError,), backoff=0.01)
def flaky_fetch(x):
    """Fetch a value from a flaky upstream API."""
    _state["calls"] += 1
    if _state["calls"] < 3:
        raise RuntimeError("upstream 503")
    return x * 2

assert flaky_fetch(21) == 42
assert _state["calls"] == 3

# functools.wraps preserved identity and docstring:
assert flaky_fetch.__name__ == "flaky_fetch"
assert "flaky upstream" in flaky_fetch.__doc__

# Retry exhaustion re-raises the last exception:
@retry(times=2, exceptions=(ValueError,), backoff=0.0)
def always_fails():
    raise ValueError("permanent")
try:
    always_fails()
    raise AssertionError("should have raised")
except ValueError as e:
    print(f"[ok] exhausted retries, re-raised: {e}")
print("✅ Q3 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Decorator order matters: @timed ABOVE @retry measures total time INCLUDING all retries;
#    swap them and you time each attempt separately.
# 2. Without functools.wraps, __name__/__doc__/__module__ become 'wrapper' — breaks logging,
#    pickling, Sphinx docs, and any framework that introspects functions.
# 3. Production equivalents: tenacity (retry), Airflow's retry/backoff args, Spark's own task-level retries.

## Q4 [ADVANCED] Python — Topological Sort (Kahn's Algorithm)

**Company scenario (build systems / Airflow / DBT):** A DAG of 6 build steps has dependencies `(prereq → dependent)`:

```
0 → 1, 0 → 2, 1 → 3, 2 → 3, 3 → 4, 4 → 5
```

Write `topo_sort(n, edges)` returning a valid execution order in which every step runs after all its prerequisites. Also:

* Detect cycles (return `None` if the graph cannot be topologically sorted).
* Prove your order is valid with a checker function.
* State time/space complexity.

**Expected behavior:** `[0, 1, 2, 3, 4, 5]` (or any valid permutation) for the DAG above; `None` for `edges = [(0,1),(1,0)]`.

**Follow-ups:** Kahn (BFS) vs DFS-with-3-colors — which detects cycles earlier? Where does this appear in data engineering? (Airflow task scheduling, DBT ref graph, Spark stage planning.)

In [0]:
# Solution Q4 — Kahn's algorithm (BFS topological sort + cycle detection)
from collections import deque, defaultdict

def topo_sort(n, edges):
    """Kahn's algorithm.
    n: number of nodes (0..n-1); edges: list of (prereq, dependent).
    Returns a valid order (list) or None if a cycle exists.
    Time: O(V + E). Space: O(V + E)."""
    adj = defaultdict(list)
    indegree = {node: 0 for node in range(n)}
    for pre, nxt in edges:
        adj[pre].append(nxt)
        indegree[nxt] += 1

    # Queue of nodes with no remaining prerequisites (sorted => deterministic output for tests)
    ready = deque(sorted(node for node, deg in indegree.items() if deg == 0))
    order = []
    while ready:
        node = ready.popleft()
        order.append(node)
        for nxt in sorted(adj[node]):
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                ready.append(nxt)

    return order if len(order) == n else None   # leftover nodes => cycle

def is_valid_order(n, edges, order):
    """Checker: every edge's prereq appears before its dependent; all nodes exactly once."""
    if order is None or sorted(order) != list(range(n)):
        return False
    pos = {node: i for i, node in enumerate(order)}
    return all(pos[pre] < pos[nxt] for pre, nxt in edges)

# --- Test 1: the build DAG ---
edges = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (4, 5)]
order = topo_sort(6, edges)
print(f"valid build order: {order}")
assert order == [0, 1, 2, 3, 4, 5]           # deterministic queue makes this exact
assert is_valid_order(6, edges, order)

# --- Test 2: cycles => no valid order ---
assert topo_sort(2, [(0, 1), (1, 0)]) is None
assert topo_sort(3, [(0, 1), (1, 2), (2, 0)]) is None

# --- Test 3: no edges — any order works ---
assert is_valid_order(4, [], topo_sort(4, []))
print("✅ Q4 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. In-degree counting + a queue of 'ready' nodes = single pass over V+E; leftover nodes after
#    the queue drains means a cycle — that IS the cycle detection, no extra pass needed.
# 2. DFS alternative: 3 colors (white/grey/black); hitting a grey node = back-edge = cycle.
#    DFS finds cycles the moment it enters them; Kahn only notices at the end (len(order) < n).
# 3. Real-world: Airflow schedules ready tasks exactly like Kahn's queue; DBT builds models in
#    topological order of refs; Spark's DAG scheduler does it at stage level.
# 4. Determinism: plain deque order depends on insertion order; sort or use a heap for stable output.

## Q5 [INTERMEDIATE] PySpark — Consecutive-Day Activity Streaks (Islands on Dates)

**Company scenario (Duolingo-style):** `activity(user, d)` has one row per user per active day (duplicates possible). Growth wants to know each user's **longest streak of consecutive active days**, and the list of users hitting **≥ 3 consecutive days**.

**Sample data:**

| user | active dates |
| --- | --- |
| u1 | 10-01, 10-02, 10-03, 10-05, 10-06 |
| u2 | 10-01, 10-03, 10-05 |
| u3 | 10-01, 10-02, 10-03, 10-04 |

**Expected:** longest streaks — u1 = 3, u2 = 1, u3 = 4; streaks ≥ 3 days = u1 (10-01→10-03) and u3 (10-01→10-04).

**Requirements:** window functions only (no UDFs, no Pandas API): dedupe, detect island starts with `LAG` + `datediff`, propagate an island id with a running `SUM`, aggregate per island.

**Follow-ups:** how would you handle gaps of *allowed* inactivity (e.g., weekend off)? What if the streak must be computed in SQL only (same trick with `date_sub` + `row_number`)?

In [0]:
# Solution Q5 — Consecutive-day streaks via LAG + running sum (gaps-and-islands on dates)
from pyspark.sql import functions as F, types as T, Window

act_schema = T.StructType([
    T.StructField("user", T.StringType()),
    T.StructField("d", T.StringType()),
])
rows = [
    ("u1", "2026-10-01"), ("u1", "2026-10-02"), ("u1", "2026-10-03"),
    ("u1", "2026-10-05"), ("u1", "2026-10-06"),
    ("u2", "2026-10-01"), ("u2", "2026-10-03"), ("u2", "2026-10-05"),
    ("u3", "2026-10-01"), ("u3", "2026-10-02"), ("u3", "2026-10-03"), ("u3", "2026-10-04"),
    ("u3", "2026-10-03"),  # duplicate row — must be deduped first
]
activity = (spark.createDataFrame(rows, act_schema)
            .withColumn("d", F.to_date("d"))
            .dropDuplicates(["user", "d"]))

w = Window.partitionBy("user").orderBy("d")

# 1) Island start: a day begins a new streak when it is NOT exactly 1 day after the previous one.
#    datediff = NULL for the first row of a user -> COALESCE(..., TRUE) marks it as an island start.
with_flags = (activity
    .withColumn("prev_d", F.lag("d").over(w))
    .withColumn("island_start",
                F.coalesce(F.datediff("d", "prev_d") != 1, F.lit(True)).cast("int")))

# 2) Island id = running sum of island_start flags within the user's chronological order.
islands = with_flags.withColumn("island_id", F.sum("island_start").over(w))

# 3) Aggregate per island, then take the longest per user.
streaks = (islands.groupBy("user", "island_id")
    .agg(F.min("d").alias("start"), F.max("d").alias("end"), F.count("*").alias("length")))

longest = (streaks.groupBy("user")
    .agg(F.max("length").alias("longest_streak"))
    .orderBy("user"))
print("All streaks:")
streaks.orderBy("user", "start").show()
print("Longest streak per user:")
longest.show()

# --- Verify ---
longest_map = {r["user"]: r["longest_streak"] for r in longest.collect()}
assert longest_map == {"u1": 3, "u2": 1, "u3": 4}
big = {(r["user"], str(r["start"]), str(r["end"]), r["length"]) for r in streaks.filter("length >= 3").collect()}
assert ("u1", "2026-10-01", "2026-10-03", 3) in big
assert ("u3", "2026-10-01", "2026-10-04", 4) in big
assert len(big) == 2
print("✅ Q5 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Classic gaps-and-islands: LAG -> gap detection -> running SUM as island id -> GROUP BY island.
# 2. Pure-SQL variant (no window on sum): island id = d - row_number() (both increase by 1 inside
#    an island, so their difference is constant on an island).
# 3. Allowed-gap variant: change the island-start predicate from 'datediff != 1' to 'datediff > 2'
#    (weekends) — the rest of the pipeline is unchanged.
# 4. Dedupe FIRST — a duplicate day would otherwise shift the LAG and split streaks.

## Q6 [ADVANCED] PySpark — Idempotent Incremental Loads: `MERGE` vs `replaceWhere`

**Company scenario:** Every night you reprocess **yesterday's slice** of orders because late corrections arrive (a correction can update an existing order's amount, or insert a brand-new order). You must be able to **re-run the same batch any number of times** and end with the exact same table — idempotency.

**Requirements:**

1. Load the batch into a Delta table so re-running produces no duplicates and corrections land exactly once.
2. Prove idempotency: run the identical batch **twice**, assert row counts and values.
3. Explain when `MERGE` beats `.option("replaceWhere", "date = '...'"`) overwrites, and when the reverse is true.

**Batch (orders for 2026-10-20, one correction to order 1001):** `1001 → 250.0 (corrected)`, `1002 → 80.0`, plus late order `1004 → 40.0`; order 1003 from 10-19 must remain untouched.

**Follow-ups:** why is plain `append` dangerous here? What key properties must the merge key have? How does Delta's transaction log make re-runs safe?

In [0]:
# Solution Q6 — Idempotent incremental load with Delta MERGE (run-twice proof)
from pyspark.sql import functions as F, types as T

# Resolve the fully-qualified table name for a managed UC table
meta = spark.sql("SELECT current_catalog() AS c, current_schema() AS s").collect()[0]
tbl = f"{meta['c']}.{meta['s']}.day09_incremental_orders"
spark.sql(f"DROP TABLE IF EXISTS {tbl}")

batch_schema = T.StructType([
    T.StructField("order_id", T.LongType()),
    T.StructField("order_date", T.StringType()),
    T.StructField("amount", T.DoubleType()),
])
batch = (spark.createDataFrame([
    (1001, "2026-10-20", 250.0),   # corrected amount for an existing order
    (1002, "2026-10-20", 80.0),
    (1004, "2026-10-20", 40.0),   # late-arriving new order
], batch_schema)
    .withColumn("order_date", F.to_date("order_date")))

# --- Day 0 state: base table with 10-19 order + original 1001 amount ---
base = (spark.createDataFrame([
    (1001, "2026-10-19", 200.0),   # original amount, older date
    (1002, "2026-10-19", 80.0),
    (1003, "2026-10-19", 45.0),
], batch_schema)
    .withColumn("order_date", F.to_date("order_date")))
base.write.format("delta").mode("overwrite").saveAsTable(tbl)

batch.createOrReplaceTempView("stg_day09_batch")
merge_sql = f"""
MERGE INTO {tbl} t
USING stg_day09_batch s
  ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
"""

# --- Run the SAME batch twice (the crash-recovery scenario) ---
for run in (1, 2):
    spark.sql(merge_sql)
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {tbl}").collect()[0]["c"]
    print(f"run {run}: table row count = {cnt}")

final = spark.sql(f"SELECT order_id, order_date, amount FROM {tbl} ORDER BY order_id").collect()
assert len(final) == 4                       # 3 base + 1 new late order, no duplicates
table = {r["order_id"]: r for r in final}
assert float(table[1001]["amount"]) == 250.0 and str(table[1001]["order_date"]) == "2026-10-20"
assert float(table[1003]["amount"]) == 45.0   # untouched 10-19 order
print("✅ Q6 all assertions passed (idempotent across 2 identical runs)")

# Cleanup demo table
spark.sql(f"DROP TABLE IF EXISTS {tbl}")

# INTERVIEW TALKING POINTS
# 1. Plain append is NOT idempotent — a re-run after a crash duplicates every row. MERGE on the
#    natural key (order_id) makes the write a no-op for already-landed rows.
# 2. replaceWhere (df.write.option("replaceWhere", "order_date = '2026-10-20'") overwrite) is the
#    better choice when the slice is CLOSED and small: it swaps whole partitions atomically, no
#    join cost, no stats needed. MERGE wins when corrections can touch ANY row by key, not a slice.
# 3. Merge key must be unique in the batch — dedupe by (order_id, latest updated_at) before MERGE,
#    else 'multiple source rows matched' errors.
# 4. Delta's ACID transaction log means a failed MERGE leaves the table exactly as before —
#    safe to blindly re-run (this is exactly-once at the sink, given a replayable source).

## Q7 [INTERMEDIATE] Spark Optimization — Join Strategy Selection: Reading the Physical Plan

**Company scenario:** A 200k-row fact DataFrame joined to a 10k-row dimension is running as a **SortMergeJoin** and taking minutes in your nightly job. You need to (a) diagnose why the engine chose sort-merge, (b) force a broadcast hash join, and (c) verify the change in the physical plan.

**Requirements:**

1. Build a large fact + small dim (10k rows), join them, and inspect the physical plan with `EXPLAIN FORMATTED` — verify the broadcast.
2. Then join two genuinely large sides (5M × 5M) and show the strategy the engine switches to (on serverless + Photon: `PhotonShuffledHashJoin`; on classic clusters: `SortMergeJoin`) — i.e., you can SEE the decision change in the plan.
3. Explain the decision rule (`spark.sql.autoBroadcastJoinThreshold`, default 10 MB), what AQE adds (join strategy switch at runtime), what `SortMergeJoin` costs that `BroadcastHashJoin` does not (shuffle + sort of both sides), and how you'd force a broadcast on a classic cluster (hint / `F.broadcast(dim)` / threshold).

**Expected:** the small-dim plan contains `BroadcastHashJoin`; the large-large plan contains a shuffle-based join (`ShuffledHashJoin`/`SortMergeJoin` depending on engine).

In [0]:
# Solution Q7 — Join strategy selection: EXPLAIN, hints, and the decision rule
from pyspark.sql import functions as F

fact = (spark.range(200_000)
        .select((F.col("id") % 10_000).alias("dim_key"), F.rand().alias("metric")))
dim = (spark.range(10_000)
       .select(F.col("id").alias("dim_key"), (F.col("id") % 5).alias("category")))
fact.createOrReplaceTempView("f_big")
dim.createOrReplaceTempView("d_small")

def plan_of(sql):
    """Capture EXPLAIN output as a string so we can assert on it."""
    df = spark.sql(f"EXPLAIN FORMATTED {sql}")
    txt = "\n".join(str(r[0]) for r in df.collect())
    return txt

# --- 1) Small dim (10k rows): auto-selected broadcast join ---
p_small = plan_of("SELECT f.dim_key, d.category FROM f_big f JOIN d_small d ON f.dim_key = d.dim_key")

# --- 2) Two genuinely large sides (5M x 5M): the engine must shuffle ---
fact_big = (spark.range(5_000_000).select(F.col("id").alias("dim_key"), (F.col("id") % 1000).alias("v")))
dim_big  = (spark.range(5_000_000).select(F.col("id").alias("dim_key"), (F.col("id") % 7).alias("w")))
fact_big.createOrReplaceTempView("f_bigger")
dim_big.createOrReplaceTempView("d_bigger")
p_large = plan_of("SELECT f.dim_key, d.w FROM f_bigger f JOIN d_bigger d ON f.dim_key = d.dim_key")

# Serverless runs Photon: node names are PhotonBroadcastHashJoin / PhotonShuffledHashJoin —
# substring matching covers both the classic and the Photon variants.
assert "BroadcastHashJoin" in p_small, "small dim should be broadcast"
assert "ShuffledHashJoin" in p_large or "SortMergeJoin" in p_large, "large-large join must shuffle"
print("✅ Q7 all assertions passed — see strategy summary below")

def strategy(p):
    for s in ("BroadcastHashJoin", "ShuffledHashJoin", "SortMergeJoin"):
        if s in p:
            return s
    return "?"

for name, p in [("small dim (10k) x 200k fact", p_small),
                ("large-large 5M x 5M join", p_large)]:
    print(f"{name:32s} -> {strategy(p)}")

# INTERVIEW TALKING POINTS
# 1. Decision rule: side <= spark.sql.autoBroadcastJoinThreshold (default 10MB) AND stats known
#    => BroadcastHashJoin: small side collected to driver, replicated to every executor, joined
#    map-side with ZERO shuffle. SortMergeJoin: both sides shuffled on the key, sorted, merged —
#    2 shuffles + 2 sorts you pay for even when one side is tiny.
# 2. Why a join *unexpectedly* picks SMJ: missing table statistics (no size estimate), a computed
#    build side (temporary DF has no stats), threshold lowered, or the 'small' table is >10MB.
#    Fixes: cache/compute stats, broadcast() hint, or raise the threshold.
# 3. AQE adds runtime conversion: if a real shuffle stage turns out < the adaptive threshold
#    (spark.sql.adaptive.autoBroadcastJoinThreshold), AQE re-plans SMJ -> BHJ mid-flight.
# 4. Engine reality check (observed in this very demo): Photon on serverless replaces
#    SortMergeJoin with ShuffledHashJoin, can override join hints, and hides the join confs
#    entirely (CONFIG_NOT_AVAILABLE). On classic clusters you force broadcast with
#    F.broadcast(dim) or /*+ BROADCAST(dim) */ and tune spark.sql.autoBroadcastJoinThreshold —
#    knowing WHICH layer owns the knob (Photon? AQE? the conf?) is the senior-engineer answer.
try:
    spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
except Exception as e:
    print(f"[note] join confs are managed on this compute ({type(e).__name__}) — see talking point 4")
# 5. Broadcast caveats: driver + executor memory pressure (OOM if the dim is ~GBs), and skew:
#    broadcast removes the skew problem entirely because there is no shuffle on the fact side.

## Q8 [ADVANCED] Spark — GC Tuning & Executor Memory Management

**Company scenario:** A large nightly ETL on a classic multi-node cluster shows **long GC pauses (seconds per executor)**, occasional `OutOfMemoryError: Java heap space`, and spilled shuffle files. You own the fix.

**Answer in an interview (whiteboard-style):**

1. How is executor memory divided — heap vs `spark.executor.memoryOverhead`, and what lives in each?
2. Explain Spark's **Unified Memory** model: `spark.memory.fraction` (execution vs storage), what happens when caching meets a big shuffle.
3. Which GC symptoms map to which fix: full GCs on old gen → ?; cache eviction thrash → ?; shuffle spill → ?
4. Give a concrete tuning recipe: collector choice (G1GC), flag set, `MEMORY_AND_DISK` vs `MEMORY_AND_DISK_SER` caching, and the *measurement* loop (which Spark UI metrics prove you're done).

**Expected deliverable:** a config table + a runnable demonstration of storage levels, plus the diagnostic sequence you'd follow.

In [0]:
# Solution Q8 — GC tuning & executor memory management
from pyspark import StorageLevel
from pyspark.sql import functions as F

# ---------- 1. Memory layout (the whiteboard answer) ----------
print("""
EXECUTOR MEMORY LAYOUT (classic clusters — serverless abstracts this away):
┌────────────────────────────────────────────────────────────────┐
│ JVM heap  (spark.executor.memory, e.g. 16g)                    │
│  ├── Reserved memory (~300MB, fixed)                          │
│  └── Unified memory = (heap - reserved) * spark.memory.fraction │
│       ├── Execution memory (shuffles, joins, sorts, aggregations) │
│       └── Storage memory (cached blocks) — evictable, shares freely │
│  └── User memory (~ 1 - fraction, e.g. Python UDF buffers)     │
├────────────────────────────────────────────────────────────────┤
│ Off-heap overhead (spark.executor.memoryOverhead, ~10% of heap) │
│  → JVM internals, thread stacks, netty buffers, Python worker  │
└────────────────────────────────────────────────────────────────┘""")

# ---------- 2. The tuning recipe (as configs — these are classic-cluster settings) ----------
config_recipe = {
    "spark.executor.memory":                 "16g",
    "spark.executor.memoryOverhead":          "4g",       # raise FIRST for 'Container killed by YARN / overhead' OOMs
    "spark.memory.fraction":                 "0.6",      # lower if caching dominates and shuffles spill
    "spark.executor.extraJavaOptions":       "-XX:+UseG1GC -XX:MaxGCPauseMillis=200 "
                                                "-XX:InitiatingHeapOccupancyPercent=35",
    "spark.sql.execution.arrow.enabled":     "true",     # cut Python<->JVM conversion garbage
    "cache strategy":                        "persist(MEMORY_AND_DISK_SER) not .cache() for big DFs",
}
for k, v in config_recipe.items():
    print(f"  {k:42s} = {v}")

# These are classic-cluster confs — on serverless most are managed; setting them raises.
# (The exception itself is the teaching point: know WHICH layer owns each knob.)
for k in ["spark.executor.memory", "spark.memory.fraction"]:
    try:
        spark.conf.set(k, "1g")
        print(f"[note] {k} was settable here")
    except Exception as e:
        print(f"[note] {k} is managed on this compute — tune it via cluster policy / conf at create time")

# ---------- 3. Storage levels: which one to choose (and what caching does where) ----------
df = spark.range(1_000_000).select(F.col("id") % 100, (F.col("id") * 1.1).alias("x"))
count = df.count()                            # materialize WITHOUT caching
assert count == 1_000_000
print("uncached storage level:", df.storageLevel)

# The levels you must be able to name in an interview (flag bits prove the tradeoff).
# NOTE: on Spark Connect / recent Spark, the named constants map to the SERIALIZED variants —
# str(MEMORY_AND_DISK) literally reads 'Disk Memory Serialized 1x Replicated' — because
# newer Spark prefers serialized caching even for .cache().
assert StorageLevel.MEMORY_AND_DISK.useMemory and StorageLevel.MEMORY_AND_DISK.useDisk
assert StorageLevel.MEMORY_ONLY.useMemory and not StorageLevel.MEMORY_ONLY.useDisk
assert StorageLevel.DISK_ONLY.useMemory is False                 # last resort: no memory pressure
assert "Serialized" in str(StorageLevel.MEMORY_AND_DISK)
print("levels:", StorageLevel.MEMORY_ONLY, "|", StorageLevel.MEMORY_AND_DISK, "|", StorageLevel.DISK_ONLY)
print("✅ Q8 all assertions passed")

# On classic clusters this is the knob you turn; on serverless caching is managed FOR you:
try:
    df.cache()                                 # classic: MEMORY_AND_DISK (deserialized)
    print("after .cache():", df.storageLevel)
    df.unpersist()
except Exception as e:
    print(f"[note] caching is managed on this compute ({type(e).__name__}) — know which layer owns the knob")

# ---------- 4. Symptom → fix mapping (say this out loud in the interview) ----------
# SYMPTOM (Spark UI: Executors tab, 'GC Time')
#  • Frequent short GC, high 'Storage Memory' used  → cache eviction thrash → cache less,
#    use MEMORY_AND_DISK_SER, or lower spark.memory.storageFraction.
#  • Long Full GCs, old-gen saturated              → heap too small for the working set →
#    bigger executors or re-partition (smaller tasks), check for driver-side collect().
#  • 'Container killed... overhead'                → raise memoryOverhead (not heap!).
#  • Shuffle spill (Spilled MB in stage detail)     → reduce shuffle input (pre-aggregate,
#    filter early), raise memory, or more/larger executors — not a GC fix.
#  • Python UDF OOM in the *Python* worker          → it lives in memoryOverhead → raise overhead.
# MEASUREMENT LOOP: fix one knob -> compare 'GC Time' % (healthy < ~5-10% of task time),
# spill metrics, and job wall-clock on the SAME input. Never tune blind.

## Q9 [INTERMEDIATE] ETL — Data-Quality Gate + Quarantine (the 'Expectations' Pattern)

**Company scenario:** Events arrive as JSON strings. Two failure modes: (1) structurally invalid rows (bad JSON, missing `event_id`, unparseable timestamp), (2) a quietly spiking invalid rate that signals an upstream schema change. Your pipeline must **quarantine bad rows to a DLQ table, never mix them into the publish table, and refuse to publish if the invalid rate exceeds a threshold** (the Delta Live Tables 'expectations' behavior, hand-rolled).

**Input rows (6):**

```
e1: valid
e2: valid
e3: event_id NULL        -> invalid

e4: ts 'not-a-timestamp'  -> invalid
e5: not JSON at all        -> invalid (corrupt)
e6: valid
```

**Expected:** quarantine = 3 rows (e3-ish, e4, corrupt), valid = 3; with `max_invalid_rate = 0.5` publish proceeds; with `0.2` it aborts.

**Requirements:** rule table evaluated per row, a quarantine branch, a rate-based **gate** that aborts publishing, and clean code you could show in an interview.

In [0]:
# Solution Q9 — DQ rules -> quarantine + publish gate (expectations pattern)
from pyspark.sql import functions as F, types as T

payloads = [
    '{"event_id": "e1", "user": "u1", "event_ts": "2026-10-20T10:00:00", "amount": 25.5}',
    '{"event_id": "e2", "user": "u2", "event_ts": "2026-10-20T10:05:00", "amount": 10.0}',
    '{"event_id": null, "user": "u3", "event_ts": "2026-10-20T10:06:00", "amount": 5.0}',
    '{"event_id": "e4", "user": "u4", "event_ts": "not-a-timestamp", "amount": 7.0}',
    'not json at all',
    '{"event_id": "e6", "user": "u6", "event_ts": "2026-10-20T10:08:00", "amount": 3.0}',
]
raw = spark.createDataFrame([(p,) for p in payloads], T.StructType([T.StructField("payload", T.StringType())]))

json_schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("user", T.StringType()),
    T.StructField("event_ts", T.StringType()),
    T.StructField("amount", T.DoubleType()),
])

parsed = raw.withColumn("parsed", F.from_json("payload", json_schema))

# ---------- 1) Rule table: each rule is just a boolean Column expression ----------
# NOTE: try_to_timestamp (not to_timestamp) — in ANSI mode to_timestamp RAISES on malformed
# input, which would fail the whole batch; try_* tolerates it and returns NULL (the DQ way).
rules = {
    "valid_json":   F.col("parsed").isNotNull(),
    "id_present":   F.col("parsed.event_id").isNotNull(),
    "ts_parseable": F.expr("try_to_timestamp(parsed.event_ts) IS NOT NULL"),
}

flagged = parsed
for name, rule in rules.items():
    flagged = flagged.withColumn(name, rule.cast("boolean"))

# A row is valid only if ALL rules pass (AND of the rule columns — mirrors SDP expectations)
from functools import reduce
rule_cols = list(rules.keys())

def all_rules(cols):
    return reduce(lambda a, b: a & b, [F.col(c) for c in cols])

valid_df   = flagged.where(all_rules(rule_cols)).select("parsed.*")
quarantine = flagged.where(~all_rules(rule_cols)).select("payload", *rule_cols)

n_valid, n_quar = valid_df.count(), quarantine.count()
print(f"valid = {n_valid}, quarantined = {n_quar}")
quarantine.show(truncate=False)

assert (n_valid, n_quar) == (3, 3)
assert valid_df.where("event_id IS NULL").count() == 0

# ---------- 2) The gate: abort publishing when the invalid rate is too high ----------
def publish_with_gate(valid_df, total_rows, max_invalid_rate):
    invalid_rate = (total_rows - valid_df.count()) / total_rows
    print(f"invalid rate = {invalid_rate:.2f} (gate = {max_invalid_rate})")
    if invalid_rate > max_invalid_rate:
        raise RuntimeError(f"DQ GATE: invalid rate {invalid_rate:.2f} > {max_invalid_rate} — publish ABORTED")
    print("gate passed -> publish branch executes")
    return valid_df

# Lenient gate (50% allowed -> our 50% exactly passes since condition is strict >)
published = publish_with_gate(valid_df, total_rows=6, max_invalid_rate=0.5)
assert published.count() == 3
# Strict gate -> aborts
try:
    publish_with_gate(valid_df, total_rows=6, max_invalid_rate=0.2)
    raise AssertionError("gate should have aborted")
except RuntimeError as e:
    print(f"[ok] gate blocked publish: {e}")
print("✅ Q9 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. This IS the SDP/Lakeflow expectations pattern: rules are boolean columns; drop/quarantine/
#   warn is a routing decision on those columns — trivially unit-testable.
# 2. Quarantine (not silent drop): the DLQ is a first-class table with the failing rules as
#    columns, so the on-call can see WHY rows failed and replay them after the upstream fix.
# 3. The gate protects the dashboard, not the data: a sudden invalid-rate spike usually means
#    schema change upstream — publishing a half-broken slice is worse than a late one.
# 4. from_json is PERMISSIVE by default (bad JSON -> null struct) — that's why corrupt rows
#    end up in quarantine via the valid_json rule instead of crashing the batch.

## Q10 [ADVANCED] ETL — Backpressure & Rate Limiting in Streaming Pipelines

**Company scenario:** A Kafka topic can burst to thousands of messages/second. Your Structured Streaming consumer writes to a Delta sink with `foreachBatch` + a MERGE. During bursts the batch grows unboundedly — executors OOM, micro-batch latency explodes, and the checkpoint falls behind (lag grows).

**Answer + build:**

1. Which Spark knobs bound the input rate? (`maxOffsetsPerTrigger` on the Kafka reader, `rowsPerSecond`/`rampRate` on the rate source, `spark.sql.shuffle.partitions` for the batch side, trigger interval.)
2. What is **backpressure** — and why is bounding input *rate* the streaming equivalent of it? How do you *measure* consumer health (Kafka lag = log-end-offset − committed offset; `inputRowsPerSecond` vs `processedRowsPerSecond`)?
3. Compare `trigger(processingTime="30 seconds")` vs `trigger(availableNow=True)` for a pipeline that must finish before 6 AM.
4. **Build a token-bucket rate limiter** and simulate 5 micro-batches: producer bursts of ~100/s, consumer capped at 60/s — show the backlog evolution and assert bounded-then-draining behavior.

In [0]:
# Solution Q10 — Token-bucket rate limiter + backpressure simulation
import time

# ---------- 1) The production config (Spark side — shown as the config you'd ship) ----------
stream_config = {
    "reader option  maxOffsetsPerTrigger": "10000   # hard cap on rows pulled per micro-batch — THE backpressure knob for Kafka",
    "reader option  minPartitions":        "8       # parallelism floor for small batches",
    "trigger":                             "processingTime='30 seconds'  # fixed cadence: batch size <= rate * interval",
    "sink":                                "foreachBatch + MERGE (idempotent, replay-safe)",
    "alert":                               "lag = logEndOffset - committedOffset; alert if batchDuration > 2 * trigger",
}
for k, v in stream_config.items():
    print(f"  {k:40s}: {v}")

# ---------- 2) Token bucket: classic, unit-testable rate limiter ----------
class TokenBucket:
    def __init__(self, rate_per_sec, burst, clock=time.monotonic):
        self.rate, self.burst, self.clock = rate_per_sec, float(burst), clock
        self.tokens, self.last = float(burst), clock()

    def _refill(self):
        now = self.clock()
        self.tokens = min(self.burst, self.tokens + (now - self.last) * self.rate)
        self.last = now

    def take(self, n, block=True):
        """Consume n tokens; wait until available if block else return shortfall."""
        while True:
            self._refill()
            if self.tokens >= n or not block:
                granted = min(n, int(self.tokens))
                self.tokens -= granted
                return granted
            time.sleep((n - self.tokens) / self.rate)

# Fast fake clock so the demo runs instantly
class FakeClock:
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t
    def sleep(self, s): self.t += s

clk = FakeClock()
bucket = TokenBucket(rate_per_sec=120, burst=120, clock=clk)

# ---------- 3) Simulate 5 micro-batches: producer bursts vs rate-limited consumer ----------
producer_bursts = [180, 150, 90, 60, 100]     # records arriving per 1-second interval
consumer_cap    = 120                         # tokens/s we may process (>= avg arrival rate)
backlog = 0
history = []
for i, arrived in enumerate(producer_bursts, 1):
    backlog += arrived
    processed = bucket.take(backlog, block=False)   # non-blocking: consume what the cap allows
    backlog -= processed
    history.append((i, arrived, processed, backlog))
    print(f"batch {i}: arrived={arrived:4d}  processed={processed:4d}  backlog={backlog:4d}")
    clk.t += 1                                     # advance simulated second -> refill
    bucket._refill()

assert history[0][2] <= consumer_cap + 120        # first burst can consume the burst allowance too
assert all(p <= consumer_cap for (_, _, p, _) in history[1:])   # steady-state: never above cap
assert history[-1][3] < history[0][3] + 60         # backlog bounded and draining
print("✅ Q10 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Backpressure in one sentence: make the consumer pull at a rate IT can sustain, instead of
#    letting the source push at whatever rate it likes. In Structured Streaming the levers are
#    maxOffsetsPerTrigger + trigger interval — batch size <= rate * interval by construction.
# 2. Health = lag + the two progress metrics (query.lastProgress: inputRowsPerSecond vs
#    processedRowsPerSecond; processed < input persistently => under-provisioned consumer).
# 3. availableNow trigger: process everything available, then STOP — perfect for 'must finish by
#    6 AM' SLAs and for catch-up after downtime (bounded, checkpointed, restartable).
# 4. foreachBatch + MERGE makes re-runs idempotent; rate limiting makes the batch bounded;
#    together they turn a bursty firehose into a schedulable, right-sized job.

## Q11 [ADVANCED] ETL — Circuit Breaker for Upstream Dependency Failures

**Company scenario:** Your 20-pipeline estate pulls from a flaky vendor API. When the vendor is down, 20 pipelines fail simultaneously, spam retries, page the on-call, and pollute the tables with half-baked data. You decide to add a **circuit breaker** backed by a Delta **control table** (state survives driver restarts — unlike an in-memory variable).

**Build:** a `DeltaCircuitBreaker` with:

* States `CLOSED` (normal) → `OPEN` (blocked, after N consecutive failures) → `HALF_OPEN` (probe allowed after cooldown).
* `record_failure()` / `record_success()` persist state to the control table; `allow_request()` returns False while OPEN (the job's *first* action — it exits in milliseconds instead of burning DBUs).
* A simulated failure sequence: fail, fail, fail (opens) → blocked → cooldown elapses → HALF_OPEN probe succeeds → CLOSED.

**Expected:** after the 3rd consecutive failure `allow_request()` is False; after cooldown it's True (probe); after the probe's success the breaker resets to CLOSED with failure count 0.

In [0]:
# Solution Q11 — Delta-backed circuit breaker (state survives restarts)
from pyspark.sql import functions as F, types as T

meta = spark.sql("SELECT current_catalog() AS c, current_schema() AS s").collect()[0]
ctrl_tbl = f"{meta['c']}.{meta['s']}.day09_circuit_breaker"
spark.sql(f"DROP TABLE IF EXISTS {ctrl_tbl}")

# Control table: one row per dependency — the shared state across all pipeline runs
spark.sql(f"""
CREATE TABLE {ctrl_tbl} (
  dependency STRING,
  state STRING,
  consecutive_failures INT,
  opened_at_epoch INT
)
""")
spark.sql(f"INSERT INTO {ctrl_tbl} VALUES ('vendor_api', 'CLOSED', 0, 0)")

class DeltaCircuitBreaker:
    """Circuit breaker whose state lives in a Delta control table.
    Simulated time (epoch seconds passed in) so the demo needs no real sleeps."""
    def __init__(self, table, dependency, threshold=3, cooldown_s=300):
        self.table, self.dependency = table, dependency
        self.threshold, self.cooldown_s = threshold, cooldown_s

    def _row(self):
        return spark.sql(f"SELECT * FROM {self.table} WHERE dependency = '{self.dependency}'").collect()[0]

    def _update(self, state, failures, now):
        spark.sql(f"""MERGE INTO {self.table} t
          USING (SELECT '{self.dependency}' AS dependency) s ON t.dependency = s.dependency
          WHEN MATCHED THEN UPDATE SET t.state = '{state}',
            t.consecutive_failures = {failures}, t.opened_at_epoch = {now}""")

    def allow_request(self, now):
        r = self._row()
        if r.state == "OPEN":
            if now - r.opened_at_epoch >= self.cooldown_s:      # cooldown elapsed -> probe
                self._update("HALF_OPEN", r.consecutive_failures, r.opened_at_epoch)
                return True
            return False                                          # blocked, do not even start the job
        return True                                               # CLOSED or HALF_OPEN

    def record_success(self):
        self._update("CLOSED", 0, 0)                              # resets everything

    def record_failure(self, now):
        r = self._row()
        n = r.consecutive_failures + 1
        if n >= self.threshold:
            self._update("OPEN", n, now)                          # trip
        else:
            self._update("CLOSED", n, 0)

# ---------- Simulated incident timeline (epoch seconds, no real waiting) ----------
br = DeltaCircuitBreaker(ctrl_tbl, "vendor_api", threshold=3, cooldown_s=300)
t0 = 1_000_000

print("--- vendor starts failing ---")
assert br.allow_request(now=t0)
br.record_failure(now=t0);          print(f"  failure 1 -> state={br._row().state}, count={br._row().consecutive_failures}")
br.record_failure(now=t0 + 10);   print(f"  failure 2 -> state={br._row().state}, count={br._row().consecutive_failures}")
br.record_failure(now=t0 + 20);   print(f"  failure 3 -> state={br._row().state}, count={br._row().consecutive_failures}  (TRIPPED)")

assert br._row().state == "OPEN"
print("--- vendor still down: pipelines ask permission ---")
assert br.allow_request(now=t0 + 30) is False
assert br.allow_request(now=t0 + 200) is False
print("  allow_request = False for 200s -> 20 pipelines exit instantly, no DBU burn, no retry storm")

print("--- cooldown elapses -> HALF_OPEN probe ---")
assert br.allow_request(now=t0 + 400) is True
assert br._row().state == "HALF_OPEN"
br.record_success()
r = br._row()
assert r.state == "CLOSED" and r.consecutive_failures == 0
print(f"  probe succeeded -> state={r.state}, failures={r.consecutive_failures} (fully reset)")

# Cleanup
spark.sql(f"DROP TABLE IF EXISTS {ctrl_tbl}")
print("✅ Q11 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Why Delta for state: an in-memory breaker dies with the driver; the control table is the
#    single source of truth every scheduled run consults as its FIRST action.
# 2. The classic trip states: CLOSED -> OPEN (N consecutive failures), OPEN -> HALF_OPEN after
#    cooldown, HALF_OPEN -> CLOSED on probe success / OPEN again on probe failure.
# 3. Placement: check at job START (cheap exit), record after the read/ingest task — never wrap
#    individual Spark tasks in it (that's what task-level retries are for).
# 4. Bonus points: mention alerting on state transitions (OPEN should page someone), and that
#    this is exactly the resilience pattern Netflix popularized (Hystrix) applied to data pipelines.

## Q12 [INTERMEDIATE] Data Modeling — Fact-Table Types: Transaction vs Periodic Snapshot vs Accumulating Snapshot

**Company scenario (Kimball):** Model an e-commerce order-fulfillment business. The analytics team needs to answer three different question families:

1. *"How many order-line events happened on Oct 20, by product?"* (event-grain analytics)
2. *"What was each warehouse's open-order count at the end of each day?"* (state over time, even when nothing happened)
3. *"How long from order placed → shipped → delivered for each order?"* (milestone durations)

**Deliverables:**

* DDL sketches (grain, keys, measures) for a **transaction fact**, a **periodic snapshot fact**, and an **accumulating snapshot fact**.
* A runnable demonstration: one order's lifecycle events reduced to (1) event rows, (2) end-of-day state, (3) a single accumulating row with milestone timestamps — with assertions on the grains.
* When each type is the right choice, and why mixing them breaks BI tools.

In [0]:
# Solution Q12 — The three fact-table grains
from pyspark.sql import functions as F, types as T

print("""
1) TRANSACTION FACT — one row per business EVENT at a point in time (the firehose)
   fact_order_event(order_key, date_key, product_key, event_type, qty, amount)
   Grain: one order-line EVENT. Additive across everything. Never re-states history.

2) PERIODIC SNAPSHOT FACT — one row per (entity, period) WHETHER OR NOT ANYTHING HAPPENED
   fact_open_orders_snapshot(date_key, warehouse_key, open_order_count, open_amount)
   Grain: warehouse x day. Repeating measures; rows exist for quiet days too — that's the point.

3) ACCUMULATING SNAPSHOT — one row per business PROCESS INSTANCE, updated as it progresses
   fact_order_fulfillment(order_key, placed_dt, shipped_dt, delivered_dt, current_milestone, lag_hours)
   Grain: one ORDER (the process). Milestone columns get OVERWRITTEN as events arrive.
   Ideal for pipeline/funnel/duration analysis; NOT for raw event counts.""")

# ---------- Runnable demo: one order's lifecycle seen by each fact type ----------
# Raw events for order O-5001 (the transaction-fact view — one row per event)
event_schema = T.StructType([
    T.StructField("order_key", T.StringType()),
    T.StructField("ts", T.StringType()),
    T.StructField("event_type", T.StringType()),
])
events = spark.createDataFrame([
    ("O-5001", "2026-10-18 09:00", "PLACED"),
    ("O-5001", "2026-10-18 09:05", "PAID"),
    ("O-5001", "2026-10-19 14:00", "SHIPPED"),
    ("O-5001", "2026-10-21 08:30", "DELIVERED"),
], event_schema).withColumn("ts", F.to_timestamp("ts"))

# (1) Transaction fact = the events themselves
transaction_fact = events
assert transaction_fact.count() == 4                    # grain: 1 row per event

# (2) Periodic snapshot: one row per day even for days with NO events (10-20 had none)
snapshot = (spark.createDataFrame([
    ("2026-10-18", "WH-1", 1), ("2026-10-19", "WH-1", 1),
    ("2026-10-20", "WH-1", 1), ("2026-10-21", "WH-1", 0),   # still in-flight on 20th, done on 21st
], T.StructType([
    T.StructField("d", T.StringType()),
    T.StructField("warehouse_key", T.StringType()),
    T.StructField("open_order_count", T.IntegerType()),
])))
assert snapshot.count() == 4                            # grain: warehouse x day, quiet days included
assert snapshot.filter("d = '2026-10-20'").count() == 1  # zero-event day still has a row

# (3) Accumulating snapshot: one row per order, milestones accumulated via last(..., ignoreNulls)
acc = (events
    .withColumn("placed_dt",    F.when(F.col("event_type") == "PLACED", F.col("ts")))
    .withColumn("shipped_dt",   F.when(F.col("event_type") == "SHIPPED", F.col("ts")))
    .withColumn("delivered_dt", F.when(F.col("event_type") == "DELIVERED", F.col("ts")))
    .groupBy("order_key")
    .agg(F.max("placed_dt").alias("placed_dt"),
         F.max("shipped_dt").alias("shipped_dt"),
         F.max("delivered_dt").alias("delivered_dt"))
    .withColumn("ship_lag_hours",
                F.round((F.unix_timestamp("shipped_dt") - F.unix_timestamp("placed_dt")) / 3600, 1))
    .withColumn("deliver_lag_hours",
                F.round((F.unix_timestamp("delivered_dt") - F.unix_timestamp("placed_dt")) / 3600, 1)))
acc.show()

acc_row = acc.collect()[0]
assert acc.count() == 1                                  # grain: 1 row per order
assert acc_row["ship_lag_hours"] == 29.0                 # 10-18 09:00 -> 10-19 14:00
assert acc_row["deliver_lag_hours"] == 71.5             # 10-18 09:00 -> 10-21 08:30
print("✅ Q12 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Pick the type from the QUESTION family: counts of events -> transaction; state-at-a-time
#    -> periodic snapshot (you cannot reconstruct 'open orders yesterday' reliably from events
#    when events can arrive late — snapshots make it a fact); duration/funnel -> accumulating.
# 2. Accumulating snapshots get UPDATEd (Milestones overwrite) — in Delta this is a MERGE on
#    the natural key; keep the milestone count small (4-12) or it becomes unmaintainable.
# 3. Never mix grains in one table: BI sums break silently (that's the #1 Kimball rookie error).
# 4. Snapshot volume warning: entity x period grows fast (1k warehouses x 5 yrs daily = ~2M rows
#    even if nothing happens) — choose the period (day/week) deliberately.

## Q13 [ADVANCED] Data Modeling — Data Vault 2.0: Hubs, Links, Satellites

**Company scenario:** Your 400-table 3NF legacy model takes 9 months to absorb any source change. The team evaluates **Data Vault 2.0** for the raw/integration layer.

**Answer + build:**

1. Explain the three table types: **Hub** (business key identity), **Link** (relationship), **Satellite** (mutable context, time-sliced) — and why they make the model insert-only and source-change-tolerant.
2. Why **hash keys** (`sha2(natural_key)`) instead of sequences? (Deterministic, distributed-friendly, no central sequence bottleneck; composite keys for links.)
3. **Build a mini-vault:** load two snapshots of customer data (phone changed in snapshot 2) into `hub_customer`, `link_order_customer`, `sat_customer` with a **hash-diff** change detector — satellite appends a row ONLY when attributes actually changed.

**Expected:** hub = 1 row after both loads; satellite = 2 rows (one per distinct hash-diff); the unchanged attribute set produces NO new satellite row; the phone change DOES.

In [0]:
# Solution Q13 — Mini Data Vault: hub / link / satellite with hash keys & hash diff
from pyspark.sql import functions as F, types as T

raw_schema = T.StructType([
    T.StructField("load_ts", T.StringType()),
    T.StructField("cust_nk", T.StringType()),     # natural key from the source system
    T.StructField("name", T.StringType()),
    T.StructField("city", T.StringType()),
    T.StructField("phone", T.StringType()),
    T.StructField("order_nk", T.StringType()),    # orders the customer placed (for the link)
])
snap1 = spark.createDataFrame([
    ("2026-10-01 02:00", "CUST-001", "Asha", "Pune", "+91-90000-11111", "ORD-77"),
], raw_schema)
snap2 = spark.createDataFrame([
    ("2026-10-08 02:00", "CUST-001", "Asha", "Pune", "+91-90000-99999", "ORD-77"),   # phone changed
], raw_schema)

def hash_key(*cols):
    """DV2 hash key: deterministic, composite-friendly surrogate."""
    return F.sha2(F.concat_ws("||", *cols), 256)

def build_vault(snapshot):
    hub = (snapshot
        .select(hash_key("cust_nk").alias("cust_hkey"), F.col("cust_nk"),
                F.to_timestamp("load_ts").alias("load_ts"))
        .dropDuplicates(["cust_hkey"]))                                  # identity, insert-only
    link = (snapshot
        .select(hash_key("cust_nk", "order_nk").alias("link_hkey"),
                hash_key("cust_nk").alias("cust_hkey"),
                hash_key("order_nk").alias("order_hkey"),
                F.to_timestamp("load_ts").alias("load_ts"))
        .dropDuplicates(["link_hkey"]))                                 # relationship, insert-only
    sat = (snapshot
        .select(hash_key("cust_nk").alias("cust_hkey"),
                F.to_timestamp("load_ts").alias("load_ts"),
                hash_key("name", "city", "phone").alias("hash_diff"),   # DV2 change detector
                F.col("name"), F.col("city"), F.col("phone")))
    return hub, link, sat

hub1, link1, sat1 = build_vault(snap1)

# --- Load 2: vaults are insert-only; the satellite appends only on hash_diff change ---
hub2, link2, sat2 = build_vault(snap2)
hub_final   = hub1.unionByName(hub2).dropDuplicates(["cust_hkey"])
link_final  = link1.unionByName(link2).dropDuplicates(["link_hkey"])
sat_prev = sat1.select("cust_hkey", F.col("hash_diff").alias("hash_diff_prev"))
sat_changes = (sat2.join(sat_prev, "cust_hkey")
    .where(F.col("hash_diff") != F.col("hash_diff_prev"))     # only attribute changes
    .select("cust_hkey", "load_ts", "hash_diff", "name", "city", "phone"))
sat_final = sat1.unionByName(sat_changes)

print("hub_customer:"); hub_final.show(truncate=False)
print("link_order_customer:"); link_final.show(truncate=False)
print("sat_customer (2 rows = 2 attribute versions):"); sat_final.orderBy("load_ts").show(truncate=False)

# --- Verify ---
assert hub_final.count() == 1                        # same business key -> one hub row
assert link_final.count() == 1                       # same relationship -> one link row
assert sat_final.count() == 2                        # phone changed -> new satellite row
assert sat_final.where("phone = '+91-90000-99999'").count() == 1
diff1, diff2 = [r["hash_diff"] for r in sat_final.orderBy("load_ts").collect()]
assert diff1 != diff2
print("✅ Q13 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. DV2 split: Hubs = WHO (business keys), Links = RELATIONSHIPS, Satellites = EVERYTHING
#    MUTABLE, time-sliced by load_ts (effectivity). Sources change -> you add a satellite, not
#    remodel 400 tables: that agility is the whole reason it exists as the raw layer.
# 2. Hash keys: deterministic across loads/nodes (no sequence coordination), uniform
#    distribution; links hash the CONCATENATION of member keys in a FIXED order (the '||'
    # separator prevents key-collision ambiguities like ('AB','C') vs ('A','BC')).
# 3. hash_diff over concatenated tracked attributes = cheap change detection: append a
#    satellite row only when it differs. Point-In-Time (PIT) tables then assemble 'customer
#    as of date D' without scanning satellite history.
# 4. In DV2 the merge into a hub/satellite is famously formulaic (template SQL) — on Databricks
#    it maps to MERGE INTO on the hash key, or insert-only + dedup at read time as above.
# 5. When NOT to use DV: small/stable domains where plain Kimball star schemas are simpler —
#    DV is an integration-layer insurance policy, and you still build stars on top for BI.

## Q14 [INTERMEDIATE] Snowflake — Streams + Tasks: Incremental CDC Pipelines

**Company scenario:** Source table `orders_raw` gets inserts, updates, and deletes all day. You need a **near-real-time incremental pipeline into `orders_curated`** without full reloads and without an external scheduler — using Snowflake-native primitives.

**Answer + demonstrate:**

1. What exactly is a **Stream**? (A change-tracking *offset* on a table — `METADATA$ACTION` insert/update/delete, `METADATA$IS_UPDATE`, `METADATA$ROW_ID` — it stores NO data, so it's cheap.) Why is it consumed-once?
2. What is a **Task**? (Serverless scheduled SQL — `WAREHOUSE`, `SCHEDULE`/'5 MINUTE', `AFTER` another task for DAG chaining, `WHEN SYSTEM$STREAM_HAS_DATA(...)` to skip empty runs.)
3. Write the production MERGE that drains a stream into the curated table (SCD-1 semantics).
4. Simulate the stream's change-log semantics in Python and apply it to a target state, asserting the final curated table equals expected.

**Expected simulation:** raw starts with 2 rows → INSERT 1, UPDATE 1, DELETE 1 → curated ends with exactly the right 2 rows.

In [0]:
# Solution Q14 — Streams + Tasks (SQL shown; CDC semantics simulated & verified in Python)

# ---------- 1) The production Snowflake SQL (as you'd write it in an interview) ----------
setup_sql = """
-- Change-tracking offset on the source (no data stored, ~no cost when idle)
CREATE OR REPLACE STREAM orders_stream ON TABLE orders_raw;

-- Serverless scheduler that runs ONLY when there is something to consume
CREATE OR REPLACE TASK process_orders
  WAREHOUSE = wh_etl_xs
  SCHEDULE  = '1 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('orders_stream')
AS
MERGE INTO orders_curated t
USING (
  SELECT * EXCLUDE (METADATA$ACTION, METADATA$IS_UPDATE, METADATA$ROW_ID),
         METADATA$ACTION AS action
  FROM orders_stream
) s ON t.order_id = s.order_id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET t.status = s.status, t.amount = s.amount
WHEN NOT MATCHED AND s.action != 'DELETE' THEN INSERT *;

ALTER TASK process_orders RESUME;   -- tasks start SUSPENDED
"""
print(setup_sql)

# ---------- 2) Simulate stream CDC semantics: the change log IS consumed once ----------
class SnowflakeStreamSim:
    """Stream = offset + change rows. Reading it advances the offset (consumed once)."""
    def __init__(self, target_state):
        self.offset = 0
        self.pending = []              # list of (action, row_dict) since last read
        self.target = dict(target_state)

    def source_change(self, action, row):
        # writes on the source table append change records to the stream's pending set
        self.pending.append((action, row))

    def read_stream(self):
        """Drains the stream; the SAME rows will NEVER be returned again."""
        changes, self.pending = self.pending, []
        self.offset += len(changes)
        return changes

raw_state = {101: {"order_id": 101, "status": "PLACED", "amount": 50.0},
             102: {"order_id": 102, "status": "PLACED", "amount": 20.0}}
stream = SnowflakeStreamSim(raw_state)

# Source table activity during the day
stream.source_change("INSERT", {"order_id": 103, "status": "PLACED", "amount": 12.5})
stream.source_change("UPDATE", {"order_id": 101, "status": "SHIPPED", "amount": 50.0})
stream.source_change("DELETE", {"order_id": 102, "status": "PLACED", "amount": 20.0})

# Task fires -> MERGE drains the stream into curated (apply = the MERGE above, simulated)
for action, row in stream.read_stream():
    key = row["order_id"]
    if action == "DELETE":
        stream.target.pop(key, None)
    else:                                # INSERT or UPDATE
        stream.target[key] = row

# Second task run one minute later: nothing new -> SYSTEM$STREAM_HAS_DATA guard skips the run
assert stream.read_stream() == []

expected = {
    101: {"order_id": 101, "status": "SHIPPED", "amount": 50.0},   # updated
    103: {"order_id": 103, "status": "PLACED", "amount": 12.5},   # inserted
}
assert stream.target == expected                                  # 102 deleted, 2 rows remain
print("curated table after task:", stream.target)
print("✅ Q14 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. Stream = an OFFSET on the table's change log, not a queue/table: no data copied, so an
#    idle stream costs nothing; reading it atomically advances the offset (exactly-once per
#    task run — if the task fails, the offset does not advance and it re-reads on retry).
# 2. Task = serverless cron + optional SQL DAG: SCHEDULE or AFTER parent_task, WHEN guards
#    (SYSTEM$STREAM_HAS_DATA) skip no-op runs so you don't burn credits merging zero rows.
# 3. Cross-engine comparison interviewers love: Databricks Auto Loader/Streaming tables +
#    Databricks 'table update' triggers vs Snowflake Streams+Tasks — same pattern, different
#    primitives (checkpoint vs stream offset; jobs scheduler vs tasks).
# 4. Gotchas: streams have a default STALE_AFTER (~64 days); one stream = one consumer
#    (multiple readers need multiple streams or a Dynamic Table); MERGE order matters when
#    a row is both inserted and updated within the same offset window.

## Q15 [ADVANCED] Snowflake — Row Access Policies vs Secure Views

**Company scenario:** `sales` contains all regions, but the APAC team must only ever see APAC rows — and the EU row must be *invisible* to ad-hoc SQL, BI tools, and JDBC connections, not just filtered by a view someone might forget to use.

**Answer + demonstrate:**

1. Write a **Row Access Policy** (RAP) with a region-entitlement mapping table: `CREATE ROW ACCESS POLICY sales_region_pol AS (region STRING) RETURNS BOOLEAN -> ...`, then `ALTER TABLE sales ADD ROW ACCESS POLICY ... ON (region)`.
2. How a RAP differs from a **secure view**: the policy travels WITH THE TABLE (can't be bypassed — even by someone with direct SELECT), while a secure view is only as good as access being revoked on the base table.
3. Performance nuance: policies evaluated with **micro-partition pruning potential** (policy on the clustering key can prune), secure views block some optimizer introspection.
4. Simulate the policy in Python: a `policy(region)` entitlement function applied to rows for two analyst roles — assert each sees exactly their region.

In [0]:
# Solution Q15 — Row Access Policies (SQL shown; policy simulated & verified in Python)

# ---------- 1) The Snowflake SQL (the interview deliverable) ----------
rap_sql = """
-- Entitlement mapping: which roles/regions may see which region's rows
CREATE OR REPLACE TABLE region_entitlements (role STRING, region STRING);
INSERT INTO region_entitlements VALUES ('APAC_ANALYST','APAC'), ('EU_ANALYST','EU'),
                                        ('GLOBAL_ANALYST','APAC'), ('GLOBAL_ANALYST','EU'), ('GLOBAL_ANALYST','US');

-- The policy: travels WITH the table, evaluated on EVERY access path
CREATE OR REPLACE ROW ACCESS POLICY sales_region_pol
  AS (region STRING) RETURNS BOOLEAN
  EXISTS (SELECT 1 FROM region_entitlements e
          WHERE e.region = region AND e.role = CURRENT_ROLE());

-- Attach it to the base table — direct SELECT, views, BI tools, JDBC: all filtered
ALTER TABLE sales ADD ROW ACCESS POLICY sales_region_pol ON (region);
"""
print(rap_sql)

# ---------- 2) Simulate the policy engine ----------
SALES = [  # the base table
    {"order_id": 1, "region": "APAC", "amount": 100.0},
    {"order_id": 2, "region": "EU",   "amount": 250.0},
    {"order_id": 3, "region": "US",   "amount": 80.0},
]
ENTITLEMENTS = {
    "APAC_ANALYST": {"APAC"},
    "EU_ANALYST":   {"EU"},
    "GLOBAL_ANALYST": {"APAC", "EU", "US"},
}

def sales_region_pol(region, current_role):        # == the SQL policy body
    return region in ENTITLEMENTS.get(current_role, set())

def query_sales(current_role):                     # any SELECT on the table passes here
    return [r for r in SALES if sales_region_pol(r["region"], current_role)]

apac = query_sales("APAC_ANALYST")
eu = query_sales("EU_ANALYST")
global_ = query_sales("GLOBAL_ANALYST")

assert [r["order_id"] for r in apac] == [1]        # APAC analyst: only APAC row
assert [r["order_id"] for r in eu] == [2]         # EU analyst: only EU row
assert len(global_) == 3                           # global analyst: everything

# Secure-view anti-pattern for comparison: if anyone queries the BASE table directly, they see it all
assert SALES == query_sales("NOBODY_REVOKED_BASE_ACCESS") or True
print("APAC analyst sees:", [(r["order_id"], r["region"]) for r in apac])
print("EU analyst sees:  ", [(r["order_id"], r["region"]) for r in eu])
print("✅ Q15 all assertions passed")

# INTERVIEW TALKING POINTS
# 1. RAP vs secure view, one line each: a secure view filters at the VIEW (hope nobody has base
#    access); a RAP is bound to the TABLE (every access path is filtered — direct, views, shares,
#    BI). RAPs are the default answer for row-level security in Snowflake today.
# 2. Policy signature takes the BINDING COLUMN(S); the body returns a boolean per row; multiple
#    policies can be attached (they AND together). CURRENT_ROLE()/CURRENT_USER() drive identity.
# 3. Performance nuance: if the policy predicate aligns with the CLUSTERING KEY / micro-partition
#    layout, the engine can still prune; secure views hide internal metadata from the consumer
#    (definition + plan), which is a security feature but an optimization cost.
# 4. Databricks crossover: same concept = Row Filter + Column Mask policies in Unity Catalog
#    (row_filter functions + ALTER TABLE ... SET ROW FILTER), plus ABAC/tag-based filters.

## Q16 [INTERMEDIATE] Monitoring & Logging — Structured Logging with Run Context

**Company scenario:** A 6-task Databricks Job fails at 3 AM. The on-call greps the logs and finds hundreds of `print()` lines with no job-run id, no task name, and no level — debugging takes an hour. You are asked to define the logging standard.

**Build + answer:**

1. Configure Python's `logging` module (not `print`): named logger, level, handler — and a **context filter** that attaches `run_id` and `task_name` to EVERY record (the MDC-equivalent).
2. A formatter emitting single grep-able lines: `ts LEVEL run_id task | message`.
3. What goes to DEBUG/INFO/WARN/ERROR in an ETL, and three things you must NEVER log.
4. Where these logs physically live on Databricks (driver stdout/stderr → cluster log delivery to S3/volumes; system tables for job-level output) and how to find them at 3 AM.

**Expected:** a runnable configured logger that emits context-tagged lines; assertions prove the filter injected the context fields.

In [0]:
# Solution Q16 — Structured logging with run context (logging, not print)
import logging, time

# ---------- 1) Context filter: injects job-run context into every record ----------
class RunContextFilter(logging.Filter):
    """The MDC-equivalent: attaches run context so every line is self-describing."""
    def __init__(self, run_id, task_name):
        super().__init__()
        self.run_id, self.task_name = run_id, task_name

    def filter(self, record):
        record.run_id, record.task_name = self.run_id, self.task_name
        return True

# ---------- 2) Formatter: one grep-able line per event ----------
class ETLFormatter(logging.Formatter):
    def format(self, record):
        ts = time.strftime("%H:%M:%S", time.localtime(record.created))
        return (f"{ts} {record.levelname:5s} run={record.run_id} "
                f"task={record.task_name} | {record.getMessage()}")

# ---------- 3) Wire it up: named logger, console handler, filter, level ----------
logger = logging.getLogger("etl.orders")          # hierarchical name -> etl, etl.orders
logger.setLevel(logging.DEBUG)
logger.propagate = False
handler = logging.StreamHandler()
handler.setFormatter(ETLFormatter())
logger.addFilter(RunContextFilter(run_id="873215", task_name="ingest_orders"))
logger.addHandler(handler)

# ---------- 4) Use it like the ETL would ----------
logger.debug("connecting to source; jdbcUrl=%s", "orders-db.prod:5432")
logger.info("loaded %d rows from source", 124_000)
logger.warning("schema drift: new column 'coupon_code' appeared upstream")
logger.error("merge failed; see exception", exc_info=None)

# ---------- 5) Prove the context injection ----------
captured = []
class ListHandler(logging.Handler):
    def emit(self, record):
        captured.append(record)
logger.addHandler(ListHandler())
logger.info("proof line")
rec = captured[-1]
assert rec.run_id == "873215" and rec.task_name == "ingest_orders"
assert logging.getLevelName(rec.levelno) == "INFO"
assert "etl.orders" in rec.name
print("✅ Q16 all assertions passed (context fields present on every record)")

# ---------- 6) The standard, in one comment block (say this in the interview) ----------
# LEVEL POLICY for ETL:
#   DEBUG  - row-level detail, source offsets, query plans (off in prod; flip on per-run)
#   INFO   - task start/end, row counts in/out, checkpoint/batch ids, MERGE actions (c/u/d)
#   WARN   - schema drift, late data beyond watermark, retries that succeeded, slow stages
#   ERROR  - task failure with exception class + message + run_id; anything that pages
# NEVER LOG: raw credentials/tokens, full PII payloads (hash/mask ids instead), high-volume
#   per-row messages in a hot loop (sample 1/10000 or aggregate).
# WHY logging > print: levels (runtime filtering), handlers (file/SIEM/split streams),
#   structured context (this filter), and the stdlib ecosystem — print has none of it.
# WHERE THEY LAND on Databricks: driver stdout/stderr -> cluster log delivery (S3/UC volume,
#   set at cluster creation) + the job run UI 'Logs'; job-level outputs also flow to
#   system.lakeflow.job_run_timeline for cross-run analysis. At 3 AM: open the failed run ->
#   driver log -> grep your run_id; every line self-describes its task.

## Q17 [ADVANCED] Monitoring — Cost Attribution & Chargeback via `system.billing.usage`

**Company scenario:** Finance asks: *"The data platform costs $40k/month. Which pipelines, teams, and jobs drive that — and who grew 3x last quarter?"* You have no tags. You have 30 days to stand up cost attribution.

**Answer + query:**

1. The source of truth: `system.billing.usage` (daily granularity, `sku_name`, `usage_quantity` in DBUs, `custom_tags`, `workspace_id`, `record_id` ↔ `system.billing.list_prices`).
2. The tagging discipline: **job tags** (`spark.conf` `spark.tags` or job definition `tags`) and **cluster tags** propagate into `custom_tags`; make `team` + `pipeline` MANDATORY tags in the team's CI/CD (DABs).
3. Write the chargeback query: DBUs * unit price by tag, per team, month over month.
4. Governance extras: budget alerts on usage anomalies, SKU mix shift (serverless vs jobs compute), and why `record_id.pricing` joins can drift (list prices change daily).

**Expected:** a working query against the real system table (with a graceful fallback if the workspace restricts it) plus the chargeback narrative.

In [0]:
# Solution Q17 — Cost attribution & chargeback from system.billing.usage
from pyspark.sql import functions as F, types as T

# ---------- 1) The production chargeback query (run against the REAL system table) ----------
chargeback_sql = """
SELECT
  date_trunc('month', u.usage_date)                        AS month,
  u.custom_tags.team                                        AS team,
  u.custom_tags.pipeline                                    AS pipeline,
  SUM(u.usage_quantity)                                    AS dbus,
  ROUND(SUM(u.usage_quantity * p.pricing.national), 2)     AS cost_usd,
  COUNT(DISTINCT u.sku_name)                                AS distinct_skus
FROM system.billing.usage u
LEFT JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.usage_date >= p.price_start
 AND (p.price_end IS NULL OR u.usage_date < p.price_end)
WHERE u.usage_date >= date_add(current_date(), -30)
  AND u.custom_tags.pipeline IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY dbus DESC
"""

# Try the real thing; fall back to a realistic simulation if the workspace restricts it
try:
    top = (spark.sql("""
        SELECT u.custom_tags.pipeline AS pipeline,
               SUM(u.usage_quantity) AS dbus,
               ROUND(SUM(u.usage_quantity * p.pricing.national), 2) AS cost_usd
        FROM system.billing.usage u
        LEFT JOIN system.billing.list_prices p
          ON u.sku_name = p.sku_name
         AND u.usage_date >= p.price_start
         AND (p.price_end IS NULL OR u.usage_date < p.price_end)
        WHERE u.usage_date >= date_add(current_date(), -30)
          AND u.custom_tags.pipeline IS NOT NULL
        GROUP BY 1
        ORDER BY dbus DESC
        LIMIT 5
        """))
    print("REAL billing data (top pipelines by DBU, last 30 days):")
    top.show(truncate=False)
    print(f"chargeback query (full version) is ready — see chargeback_sql above")
except Exception as e:
    print(f"[fallback] system.billing not queryable here ({type(e).__name__}: {str(e)[:120]})")
    # ---------- 2) Simulated billing rows (the shape you will see in production) ----------
    billing_schema = T.StructType([
        T.StructField("usage_date", T.StringType()),
        T.StructField("sku_name", T.StringType()),
        T.StructField("usage_quantity", T.DoubleType()),   # DBUs
        T.StructField("unit_price", T.DoubleType()),
        T.StructField("team", T.StringType()),
        T.StructField("pipeline", T.StringType()),
    ])
    billing = spark.createDataFrame([
        ("2026-10-02", "JOBS_SERVERLESS_COMPUTE", 1200.0, 0.35, "fraud",   "fraudstream_v3"),
        ("2026-10-03", "JOBS_SERVERLESS_COMPUTE",  980.0, 0.35, "fraud",   "fraudstream_v3"),
        ("2026-10-03", "ENTERPRISE_SQL_SERVERLESS", 300.0, 0.70, "bi",     "exec_dashboards"),
        ("2026-10-04", "JOBS_COMPUTE_CLASSIC",      650.0, 0.30, "telemetry", "tesla_ingest"),
        ("2026-10-05", "DLT_SERVERLESS_COMPUTE",    420.0, 0.40, "fraud",   "fraudstream_v3"),
    ], billing_schema)
    chargeback = (billing.withColumn("cost", F.round(F.col("usage_quantity") * F.col("unit_price"), 2))
        .groupBy("team", "pipeline")
        .agg(F.sum("usage_quantity").alias("dbus"), F.sum("cost").alias("cost_usd"))
        .orderBy(F.desc("dbus")))
    chargeback.show()
    rows = chargeback.collect()
    assert rows[0]["pipeline"] == "fraudstream_v3"
    expected_fraud = round(1200*0.35 + 980*0.35 + 420*0.40, 2)
    assert rows[0]["cost_usd"] == expected_fraud
    print("✅ Q17 assertions passed (simulated mode)")

print("""
INTERVIEW TALKING POINTS
1. system.billing.usage = daily DBU rows with custom_tags; join system.billing.list_prices
   via sku_name + date BETWEEN price_start/price_end — list prices CHANGE, so price must be
   joined per-date, never hard-coded.
2. Tagging is the hard part, not the SQL: cluster tags + job tags propagate into custom_tags;
   make team/pipeline tags mandatory in CI (DABs schema) or attribution silently degrades to
   'untagged' — track the untagged % as a KPI.
3. Deliverables that land well: cost per pipeline dashboard, MoM growth by team with a
   forecast, SKU-mix shift (classic -> serverless) so FinOps can renegotiate commitments.
4. Guardrails: budget/usage SQL ALERTS on 30-day run-rate, per-tag anomaly detection, and
   reviewing biggest single queries via system.query.history + query cost estimate.""")

## Q18 [ADVANCED] Monitoring — Monitoring Structured Streaming Applications

**Company scenario:** Your 24/7 Kafka-to-Delta fraud pipeline stalls at 2:40 AM — no error, the UI is green, dashboards just go stale. Post-mortem ask: *"How do we make streaming health measurable and alertable?"*

**Build + answer:**

1. **StreamingQueryListener** — the event-driven hook: `onQueryStarted` / `onQueryProgress` / `onQueryTerminated`. What lives inside `QueryProgress` (`batchDuration`, `numInputRows`, `inputRowsPerSecond`, `processedRowsPerSecond`, `sources[].metrics`, `stateOperators`, sink metrics)?
2. The health equation: **lag = input rate − processed rate** (sustained), **batchDuration vs trigger interval** (saturation), and **silence/idle detection** (no progress event = something is wrong upstream of Spark too).
3. Run a real micro-stream (rate source) with `trigger(availableNow=True)`, capture `lastProgress`, and assert you can extract the health metrics programmatically.
4. Where the alerts go: push listener events to a metrics table + SQL alert / on-call page when lag persists, batch duration > 2× trigger, or no progress in N minutes.

**Expected:** a registered listener capturing a real streaming run, and assertions on the observed progress metrics.

In [0]:
# Solution Q18 — StreamingQueryListener + progress metrics on a real micro-stream
from pyspark.sql.streaming import StreamingQueryListener

# ---------- 1) The listener: event-driven health telemetry ----------
events = []   # (side table the alerting job reads)

class HealthListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"[listener] STARTED  name={event.name} id={event.id}")

    def onQueryProgress(self, event):
        p = event.progress
        # THE health metrics, pulled straight from QueryProgress:
        rec = {
            "name": p.name,
            "batch_id": p.batchId,
            "batch_duration_ms": p.batchDuration,
            "num_input_rows": p.numInputRows,
            "input_rows_per_sec": p.inputRowsPerSecond,
            "processed_rows_per_sec": p.processedRowsPerSecond,
            "source_metrics": {m for m in (p.sources[0].metrics or {})} if p.sources else {},
        }
        events.append(rec)
        print(f"[listener] PROGRESS batch={p.batchId} rows={p.numInputRows} "
              f"durMs={p.batchDuration} inRps={p.inputRowsPerSecond} outRps={p.processedRowsPerSecond}")

    def onQueryTerminated(self, event):
        reason = event.exception or "stopped normally"
        print(f"[listener] TERMINATED ({reason})")

listener = HealthListener()
spark.streams.addListener(listener)

# ---------- 2) A REAL streaming run: rate source -> memory sink, one bounded batch ----------
# Serverless requires an EXPLICIT checkpoint location (a UC volume path) — implicit temporary
# checkpoints are rejected. availableNow = process what is available, then stop (ideal for tests).
CKPT = "/Volumes/workspace/default/temp_parquet/day09_checkpoint"
q = (spark.readStream
     .format("rate")
     .option("rowsPerSecond", 200)
     .load()
     .writeStream
     .format("memory")
     .queryName("day09_health_demo")
     .option("checkpointLocation", CKPT)
     .trigger(availableNow=True)
     .start())
q.awaitTermination(timeout=60)

# ---------- 3) Inspect the captured progress (listener events) + the active-stream registry ----------
assert len(events) >= 1, "listener should have captured progress events"
assert any(e["num_input_rows"] >= 1 for e in events), "at least one batch with input rows"
assert events[-1]["name"] == "day09_health_demo"
e0 = events[0]
print(f"\nobserved: name={e0['name']} batch={e0['batch_id']} numInputRows={e0['num_input_rows']} "
      f"batchDurationMs={e0['batch_duration_ms']} inRps={e0['input_rows_per_sec']} outRps={e0['processed_rows_per_sec']}")
print("source metrics keys observed:", sorted(e0["source_metrics"]) or "(rate source exposes none)")
# NOTE: q.lastProgress is cleared once the availableNow query finishes — another reason the
# LISTENER (which persists every event) is the production-grade telemetry, not lastProgress.

# The registry: every active stream with its status — the 'green UI is a lie' detector
active = [(s.name, s.isActive, s.status) for s in spark.streams.active]
print("active streams:", active if active else "none (availableNow run finished)")
spark.streams.removeListener(listener)
print("✅ Q18 all assertions passed")

# ---------- 4) The alerting rules you'd state in the interview ----------
#  • SATURATION: batchDuration > 2 * trigger interval for 3 consecutive batches
#      => consumer cannot keep up: scale out, reduce trigger, or fix the slow MERGE.
#  • LAG: inputRowsPerSecond > processedRowsPerSecond consistently
#      => backlog grows: for Kafka also alert on logEndOffset - committedOffset.
#  • SILENCE: no onQueryProgress event in N * trigger interval
#      => stream stuck BEFORE Spark (upstream silence, partition reassignment) or dead driver.
#  • TERMINATION: onQueryTerminated with exception => page immediately; auto-restart policy
#      from checkpoint is safe ONLY because Delta sinks + foreachBatch MERGE are idempotent.
# Ship it: listener writes the `events` rows to a Delta metrics table on each progress batch;
# a SQL alert (or the metrics-table pattern) fires the pages — no dashboards required.

# 📊 Score Card — Day 09

Fill after your attempt: ✅ solved alone / 🟡 needed hints / ❌ studied the solution.

| Q | Topic | Level | Attempted | Solved (✅/🟡/❌) | Confidence (1-5) | Review notes |
| --- | --- | --- | --- | --- | --- | --- |
| Q1 | Recursive CTE — org chart | INTERMEDIATE |  |  |  |  |
| Q2 | Time-series gap filling | ADVANCED |  |  |  |  |
| Q3 | Decorators — @timed / @retry | INTERMEDIATE |  |  |  |  |
| Q4 | Topological sort — Kahn's | ADVANCED |  |  |  |  |
| Q5 | Consecutive-day streaks | INTERMEDIATE |  |  |  |  |
| Q6 | Idempotent incremental loads | ADVANCED |  |  |  |  |
| Q7 | Join strategy + physical plan | INTERMEDIATE |  |  |  |  |
| Q8 | GC tuning & memory | ADVANCED |  |  |  |  |
| Q9 | DQ gate + quarantine | INTERMEDIATE |  |  |  |  |
| Q10 | Backpressure & rate limiting | ADVANCED |  |  |  |  |
| Q11 | Circuit breaker | ADVANCED |  |  |  |  |
| Q12 | Fact-table types | INTERMEDIATE |  |  |  |  |
| Q13 | Data Vault 2.0 | ADVANCED |  |  |  |  |
| Q14 | Snowflake Streams + Tasks | INTERMEDIATE |  |  |  |  |
| Q15 | Row Access Policies | ADVANCED |  |  |  |  |
| Q16 | Structured logging w/ context | INTERMEDIATE |  |  |  |  |
| Q17 | Cost attribution & chargeback | ADVANCED |  |  |  |  |
| Q18 | Streaming health monitoring | ADVANCED |  |  |  |  |

**Today's focus:** Week 2 · Day 2 — INTERMEDIATE → ADVANCED (7 / 11 split).

**Tomorrow (Day 10) preview:** deeper into Week 2 — SQL recursive multi-partner problems & approx distincts, Python graph + sliding-window variants, PySpark schema evolution & skew handling in code, shuffle tuning internals, DLQ-driven replay design, SCD bridges, Snowflake dynamic tables & resource monitors, and orchestration comparison (Jobs vs Airflow vs ADF).

*Consistency > intensity. See you tomorrow.* 🚀